### ☀️ 참고 ☀️ 한글깨짐 방지

- 아래 코드셀 3개 실행 필수
- 윈도우, 맥 동일

❗️주의❗️
- Step1. 첫째 셀 실행
- Step2. '런타임 > 세션 다시시작'
- Step3. 이후 두번째 셀 실행

In [1]:
# 나눔 폰트 설치
# 해당 셀 실행 후 '런타임 > 세션 다시 시작'

%%capture

!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl


# 나눔고딕 폰트 사용 설정 (폰트 설치 후 실행)
plt.rcParams["font.family"]="NanumGothic"

# 마이너스 깨짐 방지 (일반 하이픈(-)을 사용하게 설정)
mpl.rcParams['axes.unicode_minus'] = False

In [ ]:
# 한글 폰트 테스트 그래프

plt.plot([1, 2, 3], [10, -20, 30])
plt.title('한글 폰트 테스트')
plt.xlabel('x축')
plt.ylabel('y축')
plt.grid(True)
plt.show()

# 📘 CH10. 특성 엔지니어링 및 데이터 수집 — 강의용 노트북

정규 4시간 수업 진행용입니다. Notion(md) 문서와 동일한 깊이로 구성했습니다.

> 📂 **실습 파일 안내**: 01(인코딩), 02(스케일링) 소단원은 CH09에서 사용한 `orders.csv`를 그대로 이어서 사용합니다.
> 03(웹크롤링)은 실제 웹사이트(books.toscrape.com — 스크래핑 연습을 위해 만들어진 공개 사이트)에 직접 요청을 보냅니다.
> 04(OpenAPI)는 실제 무료 공개 API(Open-Meteo, 인증키 불필요)를 사용합니다. **인터넷 연결이 필요합니다.**

---

# 📘 CH10-01. 인코딩 (Label, One-Hot, 응용)

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 인코딩이 필요한 이유 → Label Encoding → One-Hot Encoding → 여러 컬럼 동시 인코딩 → 새로운 범주 처리 → Label vs One-Hot 비교
> 🔧 **실습파일**: `orders.csv` 필요
> ⏱️ **예상 소요시간**: 45분
> 🎚️ **난이도**: ★★★★☆
> 🔗 **사전 지식**: CH09(데이터 전처리)

---

## 🤔 먼저 생각해보기

> **머신러닝 모델에게 "전자기기", "생활용품", "패션"이라는 글자를 그대로 학습시킬 수 있을까요?**

컴퓨터, 특히 머신러닝 모델은 근본적으로 숫자만 계산할 수 있습니다. 문자로 된 범주형 데이터를 숫자로 바꿔주는 과정이 **인코딩**입니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 준비 — CH09에서 전처리한 orders.csv 이어서 사용

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd

df = pd.read_csv("orders.csv")
# CH09에서 배운 전처리를 먼저 적용 (공백 정리, 콤마 제거, 결측치 채우기)
df["카테고리"] = df["카테고리"].str.strip()
df["단가"] = df["단가"].str.replace(",", "").astype(int)
df["카테고리"] = df["카테고리"].fillna(df["카테고리"].mode()[0])
df = df.drop_duplicates()
print(df[["상품명", "카테고리", "결제수단코드"]].head())
print(df["카테고리"].unique())          # 전자기기, 생활용품, 패션, 반려동물
```

</details>

In [ ]:
# 코드 입력

import pandas as pd

df = pd.read_csv("orders.csv")

# CH09에서 배운 전처리를 먼저 적용 (공백 정리, 콤마 제거 후 정수 변환, 결측치 채우기)
df["카테고리"] = df["카테고리"].str.strip()
df["단가"] = df["단가"].str.replace(",", "").astype(int)
df["카테고리"] = df["카테고리"].fillna(df["카테고리"].mode()[0])

df = df.drop_duplicates()
print(df[["상품명", "카테고리", "결제수단코드"]].head())
print(df["카테고리"].unique())

### 1) 인코딩이 필요한 이유

- 모델은 "전자기기"라는 글자 자체를 이해하지 못하고, 반드시 숫자로 변환된 값을 필요로 함
- 어떻게 숫자로 바꾸느냐에 따라 모델이 데이터를 "잘못 이해"할 수도 있음 (예: 순서가 없는데 순서가 있는 것처럼 인코딩)

### 2) Label Encoding — 범주마다 정수를 하나씩 부여

- 범주형 변수를 하나의 숫자 컬럼으로 변환
- 범주마다 0, 1, 2처럼 번호를 부여
- 원핫 인코딩과 달리 컬럼 수가 늘어나지 않음
- 숫자 크기에 의미가 생길 수 있으므로 순서가 없는 입력 변수에는 주의
- 주로 순서형 변수나 목표 변수(y) 변환에 사용
- fit_transform() : 범주를 학습하고 숫자로 변환
- classes_ : 어떤 순서로 번호가 부여되었는지 확인

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["카테고리_encoded"] = le.fit_transform(df["카테고리"])
print(df[["카테고리", "카테고리_encoded"]].drop_duplicates())
print(le.classes_)     # 어떤 순서로 숫자가 매겨졌는지 확인
```

</details>

In [ ]:
# 코드 입력





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# Pandas만으로도 간단히 라벨 인코딩과 유사하게 처리 가능
df["카테고리_factorize"], uniques = pd.factorize(df["카테고리"])
print(df[["카테고리", "카테고리_factorize"]].drop_duplicates())
print(uniques)
```

</details>

In [ ]:
# Pandas만으로도 간단히 라벨 인코딩과 유사하게 처리 가능

df["카테고리_factorize"], uniques = pd.factorize(df["카테고리"])
print(df[["카테고리", "카테고리_factorize"]].drop_duplicates())
print(uniques)

> ⚠️ Label Encoding은 "하=0, 중=1, 상=2"처럼 **크기 관계가 있는 순서형 데이터**에 적합합니다. `LabelEncoder`는 범주를 알파벳/가나다 순으로 임의로 숫자를 매기기 때문에, "전자기기=0, 패션=3"처럼 실제로는 순서가 없는데 모델이 "패션이 전자기기보다 크다"고 착각할 위험이 있습니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 순서를 직접 지정하고 싶다면 map()을 사용하는 것이 더 안전
# 고객 등급처럼 실제로 순서가 있는 데이터에 적용
customers = pd.read_csv("customers.csv")
grade_order = {"신규": 0, "일반": 1, "VIP": 2}
customers["등급_인코딩"] = customers["등급"].map(grade_order)
print(customers[["고객명", "등급", "등급_인코딩"]].head())
```

</details>

In [ ]:
# encoding을 map() 함수로 직접 하기
# 순서를 직접 지정하고 싶다면 map()을 사용하는 것이 더 안전
# 고객 등급처럼 실제로 순서가 있는 데이터에 적용

customers = pd.read_csv("customers.csv")

# 변수 분리 방식 (Extracted)



# 인라인 방식 (Inline)




### 3) One-Hot Encoding — 각 범주를 독립적인 0/1 컬럼으로

- 원핫인코딩 (범주형 변수를 숫자로. 대신 컬럼이 늘어남. 순서 상관없이 하게끔)
- 이진벤터 (0,1)로 변환
- 출력해보면 False, True로 변환. 이게 최신
- drop_first=True : 다중공선성 방지 (첫 번째 더미 컬럼 자동 삭제)
- dtype=float     : 결과를 float으로 통일 (버전마다 bool/int 다를 수 있음)


<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
one_hot = pd.get_dummies(df["카테고리"])
print(one_hot.head())
#    반려동물  생활용품   전자기기    패션
# 0   False   True   False  False
# 1   False  False    True  False
```

</details>

In [ ]:
# 코드 입력





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 원본 DataFrame에 바로 합치기
df_encoded = pd.get_dummies(df, columns=["카테고리"])
print(df_encoded.columns.tolist())
```

</details>

In [ ]:
# 원본 DataFrame에 바로 합치기





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# drop_first=True: 첫 번째 컬럼을 생략 (다중공선성 방지, 회귀 모델 등에서 유용)
df_encoded2 = pd.get_dummies(df, columns=["카테고리"], drop_first=True)
print(df_encoded2.columns.tolist())    # 카테고리_반려동물 컬럼이 사라짐 (나머지 3개가 모두 0이면 반려동물이라는 뜻)
```

</details>

In [ ]:
# drop_first=True: 첫 번째 컬럼을 생략 (다중공선성 방지, 회귀 모델 등에서 유용)
# 다중공선성: 독립변수(x) 사이에 강한 상관관계가 있을때 모델이 헷갈림





# 카테고리_반려동물 컬럼이 사라짐 (나머지 3개가 모두 0이면 반려동물이라는 뜻)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# sklearn의 OneHotEncoder (모델 파이프라인에 통합하기 편리)
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(df[["카테고리"]])
print(encoded[:5])
print(ohe.get_feature_names_out())
```

</details>

In [ ]:
# sklearn의 OneHotEncoder (모델 파이프라인에 통합하기 편리)



# sparse_output=False 넘파이 배열로 반환 (True일 경우 출력없이 sparse matrix 객체 반환)






### 4) 여러 컬럼 동시에 인코딩하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
df_multi = pd.get_dummies(df, columns=["카테고리", "결제수단코드"])
print(df_multi.columns.tolist())
```

</details>

In [ ]:
# 1. LabelEncoder
# 주의: LabelEncoder는 컬럼 1개씩 처리하도록 설계됨 → for문이 정석

from sklearn.preprocessing import LabelEncoder






In [ ]:
# 2. OneHotEncoder






### 5) 새로운 범주(unseen category) 처리 문제

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 훈련 데이터에는 "전자기기","생활용품","패션","반려동물"만 있었는데
# 테스트 데이터에 "도서"라는 새로운 카테고리가 등장하면?
le2 = LabelEncoder()
le2.fit(df["카테고리"])
try:
    le2.transform(["도서"])
except ValueError as e:
    print("에러 발생:", e)    # 훈련 때 못 본 값이라 변환 불가!

# One-Hot 방식은 그냥 모든 컬럼이 0인 행으로 처리되어 상대적으로 안전
new_row = pd.DataFrame({"카테고리": ["도서"]})
print(pd.get_dummies(new_row, columns=["카테고리"]))   # 컬럼 자체가 새로 생김 (기존 데이터와 구조가 달라짐)
```

</details>

In [ ]:
# 훈련 데이터에는 "전자기기","생활용품","패션","반려동물"만 있었는데
# 테스트 데이터에 "도서"라는 새로운 카테고리가 등장하면?

# LabelEncoder
le2 = LabelEncoder()
le2.fit(df["카테고리"])
le2.transform(["도서"]) # 훈련 때 못 본 값이라 변환 불가! 에러발생!

# One-Hot Encoder
# One-Hot 방식은 에러는 안남 그러나 더 큰 문제!
# 컬럼 자체가 새로 생김 (기존 데이터와 구조가 달라짐)
# Sklearn의 OneHotEncoder로 해결

new_row = pd.DataFrame({"카테고리": ["도서"]})
print(pd.get_dummies(new_row, columns=["카테고리"]))

# LabelEncoder
# 목표(Target) 변수에 많이 사용
# 트리 모델 같은 경우엔 x 에도 사용

### 6) 서로 다른 방식 비교 — Label Encoding vs One-Hot Encoding

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# Label Encoding: 순서형 데이터에 적합 (신규<일반<VIP처럼 크기 비교가 의미 있음)
grade_order = {"신규": 0, "일반": 1, "VIP": 2}
label_result = customers["등급"].map(grade_order)
print("Label Encoding 결과:")
print(label_result.head())
```

</details>

In [ ]:
# Label Encoding: 순서형 데이터에 적합 (신규<일반<VIP처럼 크기 비교가 의미 있음)





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# One-Hot Encoding: 순서 없는 데이터에 적합 (전자기기가 패션보다 "크다"는 의미가 없음)
onehot_result = pd.get_dummies(df["카테고리"])
print("One-Hot Encoding 결과:")
print(onehot_result.head())
```

</details>

In [ ]:
# One-Hot Encoding: 순서 없는 데이터에 적합 (전자기기가 패션보다 "크다"는 의미가 없음)





| 구분 | Label Encoding | One-Hot Encoding |
|---|---|---|
| 결과 형태 | 하나의 컬럼에 정수값 | 범주 개수만큼 0/1 컬럼 |
| 적합한 데이터 | 순서형(하<중<상) | 명목형(순서 없음, 전자기기/패션 등) |
| 차원 증가 | 없음 | 범주 개수만큼 컬럼 증가 |
| 잘못 쓰면? | 모델이 없는 순서를 학습할 위험 | 범주가 매우 많으면 차원 폭발 |

> 💡 **핵심 요약**: 순서가 있는 범주(등급, 학점 등)는 Label Encoding(또는 직접 매핑)을, 순서가 없는 범주(카테고리, 지역 등)는 One-Hot Encoding을 사용하세요.

---

## ⚠️ 자주 하는 실수

> ⚠️ 순서가 없는 카테고리(예: "전자기기","패션")에 `LabelEncoder`를 그대로 써서, 모델이 존재하지 않는 크기 관계를 학습하게 만드는 경우가 매우 흔합니다.

> ⚠️ `get_dummies()`를 훈련 데이터와 테스트 데이터에 각각 따로 적용해서, 두 데이터의 컬럼 구성(개수·순서)이 달라지는 문제가 생기는 경우가 있습니다 — 테스트에만 있는 범주 때문입니다.

> ⚠️ 범주 개수가 매우 많은 컬럼(예: 상품명 100개)에 One-Hot을 그대로 적용해 컬럼이 폭발적으로 늘어나는 것을 인지하지 못하는 경우가 있습니다.

---

## 🚀 실무에서는?

> 🚀 실무에서는 범주 개수가 적으면(카테고리, 결제수단 등) One-Hot을, 범주 개수가 매우 많으면(상품 ID, 우편번호 등) Label Encoding이나 Target Encoding 같은 다른 기법을 고려합니다. sklearn 파이프라인에 포함시킬 때는 `pd.get_dummies()`보다 `OneHotEncoder`가 훈련/테스트 데이터의 컬럼 구성을 자동으로 맞춰주기 때문에 더 안전합니다.

---

## 🧩 Check Point

**Q1.** 순서가 있는 범주형 데이터(예: 학점 F<D<C<B<A)에 더 적합한 인코딩 방식은? ① One-Hot Encoding ② Label Encoding ③ 둘 다 무관 ④ 인코딩 불필요
<details><summary>정답</summary>② Label Encoding</details>

**Q2.** `get_dummies()`에서 다중공선성을 방지하기 위해 첫 번째 컬럼을 생략하는 옵션은? ① `drop_last=True` ② `drop_first=True` ③ `sparse=True` ④ `first=False`
<details><summary>정답</summary>② `drop_first=True`</details>

**Q3.** 훈련 데이터에 없던 새로운 범주가 테스트 데이터에 등장하면 `LabelEncoder.transform()`은 어떻게 되나요? ① 자동으로 새 번호 부여 ② NaN 반환 ③ ValueError 에러 발생 ④ 0으로 처리
<details><summary>정답</summary>③ ValueError 에러 발생</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (Label Encoding): "등급"(신규/일반/VIP)을 직접 매핑으로 순서에 맞게 인코딩하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
grade_order = {"신규": 0, "일반": 1, "VIP": 2}
customers["등급_인코딩"] = customers["등급"].map(grade_order)
print(customers[["고객명", "등급", "등급_인코딩"]])
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 2 (One-Hot 기본): "카테고리" 컬럼을 get_dummies()로 원-핫 인코딩하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(pd.get_dummies(df["카테고리"]).head())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 3 (drop_first): 위 결과를 drop_first=True 옵션과 함께 다시 만들어 컬럼 차이를 확인하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
print(pd.get_dummies(df, columns=["카테고리"], drop_first=True).columns.tolist())
```

</details>

In [ ]:
# 코드 입력




In [ ]:
# TODO 4 (LabelEncoder): sklearn의 LabelEncoder로 "결제수단코드"를 인코딩하고 classes_를 확인하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
from sklearn.preprocessing import LabelEncoder
le3 = LabelEncoder()
df["결제수단_encoded"] = le3.fit_transform(df["결제수단코드"])
print(le3.classes_)
print(df[["결제수단코드", "결제수단_encoded"]].head())
```

</details>

In [ ]:
# 코드 입력




---

## 📝 실습 과제

In [ ]:
# 과제 1: 상품 등급 데이터 ["C","A","B","A","F"]를 F<D<C<B<A 순서에 맞게 직접 매핑으로 인코딩하세요

In [ ]:
# 과제 2: "도시" 컬럼(서울/부산/대구 등)을 get_dummies()로 원-핫 인코딩하고,
#         drop_first=True 적용 시와 미적용 시 컬럼 개수 차이를 확인하세요

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
grades = pd.Series(["C", "A", "B", "A", "F"])
grade_map = {"F": 0, "D": 1, "C": 2, "B": 3, "A": 4}
print(grades.map(grade_map))
```

</details>

In [ ]:
# 과제 1 예시 정답





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
no_drop = pd.get_dummies(customers, columns=["도시"])
with_drop = pd.get_dummies(customers, columns=["도시"], drop_first=True)
print("drop_first 미적용 컬럼 수:", no_drop.shape[1])
print("drop_first 적용 컬럼 수:", with_drop.shape[1])
```

</details>

In [ ]:
# 과제 2 예시 정답





---

## 📌 핵심 정리

- 인코딩은 문자 범주를 모델이 이해할 수 있는 숫자로 변환하는 과정
- Label Encoding: 순서가 있는 범주(등급 등)에 적합, `LabelEncoder` 또는 `map()`으로 직접 순서 지정
- `pd.factorize()`: Pandas만으로 간단히 라벨 인코딩과 유사하게 처리
- One-Hot Encoding: 순서가 없는 범주에 적합, `pd.get_dummies()` 또는 `OneHotEncoder`
- `drop_first=True`: 다중공선성 방지를 위해 첫 컬럼 생략
- 새로운 범주(unseen category)는 `LabelEncoder`에서 에러의 원인이 될 수 있음 — 처리 방식을 미리 고려해야 함

---

---

# 📘 CH10-02. 스케일링 (MinMax, Standard, Robust, 응용)

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: 스케일링이 필요한 이유 → MinMaxScaler → StandardScaler → RobustScaler → train/test 적용 시 주의점 → 세 스케일러 비교
> 🔧 **실습파일**: `orders.csv` 필요
> ⏱️ **예상 소요시간**: 45분
> 🎚️ **난이도**: ★★★★☆
> 🔗 **사전 지식**: CH10-01(인코딩)

---

## 🤔 먼저 생각해보기

> **"수량"(1-5개, 이상치 40-60개)과 "단가"(7,500원~89,000원)를 같은 저울에 놓고 "거리"를 계산하면, 단가의 숫자가 훨씬 커서 수량 차이는 거의 무시되지 않을까요?**

값의 범위(단위)가 크게 다른 컬럼들을 비슷한 규모로 맞춰주는 전처리가 **스케일링**입니다.

---

## 📖 핵심 개념

### 0) 실습 데이터 준비

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd

df = pd.read_csv("orders.csv")
df["단가"] = df["단가"].str.replace(",", "").astype(int)
df = df.drop_duplicates()
df = df.dropna(subset=["수량"])   # 스케일링 실습에서는 결측치를 우선 제외하고 시작
print(df[["상품명", "수량", "단가"]].describe())
# 수량: 1~5가 대부분인데 최댓값이 60! (CH09에서 다룬 그 이상치)
# 단가: 7,500 ~ 89,000으로 수량보다 훨씬 큰 범위
```

</details>

In [ ]:
# 코드 입력

import pandas as pd

df = pd.read_csv("orders.csv")

# 스케일링 실습에서는 결측치를 우선 제외하고 시작
df["단가"] = df["단가"].str.replace(",", "").astype(int)
df = df.drop_duplicates()
df = df.dropna(subset=["수량"])
print(df[["상품명", "수량", "단가"]].describe())

# 수량: 1~5가 대부분인데 최댓값이 60! (CH09에서 다룬 그 이상치)
# 단가: 7,500 ~ 89,000으로 수량보다 훨씬 큰 범위

### 1) 스케일링이 필요한 이유

- 거리 기반 모델(KNN 등)이나 경사하강법 기반 모델은 값의 범위 차이에 민감함
- "수량"(1-60)과 "단가"(7,500-89,000)를 그대로 쓰면, 모델은 단가 차이만 보고 수량 차이는 거의 무시하게 됨

### 2) MinMaxScaler — 값의 범위를 0~1 사이로 (표준화)

> 💡 MinMaxScaler는 값의 범위를 정확히 0~1로 고정하고 싶을 때 유용하지만, 이상치가 있으면 대부분의 값이 좁은 구간에 몰리는 문제가 있습니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df["단가_minmax"] = scaler.fit_transform(df[["단가"]])
print(df[["단가", "단가_minmax"]].describe())
# 최솟값 0, 최댓값 1로 고정됨
```

</details>

In [ ]:
# 코드 입력





### 3) StandardScaler — 평균 0, 표준편차를 1로 (정규화)

> 💡 StandardScaler는 데이터가 정규분포에 가까울 때 특히 효과적이며, 회귀 모델이나 딥러닝에서 가장 널리 쓰이는 스케일링 방식입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from sklearn.preprocessing import StandardScaler

scaler2 = StandardScaler()
df["단가_standard"] = scaler2.fit_transform(df[["단가"]])
print(df[["단가", "단가_standard"]].describe())
# 평균 0, 표준편차 1로 변환됨
```

</details>

In [ ]:
# 코드 입력






### 4) RobustScaler —  중앙값 0, IQR 기준 (이상치에 강함)

> 💡 RobustScaler는 평균/최댓값 대신 중앙값과 사분위수(IQR)를 사용하므로, CH09에서 배운 이상치의 영향을 훨씬 덜 받습니다.
- 중앙값과 IQR을 기준으로 스케일링 (이상치 때문에 정상 데이터의 스케일이 망가지지 않음)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from sklearn.preprocessing import RobustScaler

# 이상치가 포함된 "수량" 컬럼으로 비교 (CH09에서 발견한 40, 50, 60이 그대로 남아있음)
scaler3 = RobustScaler()
df["수량_robust"] = scaler3.fit_transform(df[["수량"]])
print(df[["수량", "수량_robust"]].sort_values("수량", ascending=False).head())
```

</details>

In [ ]:
# 코드 입력






### 5) fit(), transfrom() 실제 적용

- fit()은 계산만 (예: 평균키 170, 표준편차 10)
- transform은 변환만 (예: 170 → 0, 180 → 1, 160 → -1)
- fit_transform()은 계산 + 변환 동시

In [ ]:
# 훈련 데이터와 테스트 데이터를 나눴다고 가정





# ✅ 올바른 방법: train은 fit + transform, test는 transform만
# train으로 평균/표준편차 계산 + 적용



# train에서 계산한 기준을 test에 그대로 적용




print("train 평균(스케일링 후):", train_scaled.mean(axis=0).round(3))   # 거의 0
print("test 평균(스케일링 후):", test_scaled.mean(axis=0).round(3))     # 0이 아닐 수 있음 (정상)

> ⚠️ **절대 하면 안 되는 것**: 테스트 데이터에도 `fit_transform()`을 새로 적용하는 것. 이렇게 하면 테스트 데이터의 정보가 스케일링 기준에 섞여 들어가는 **Data Leakage(데이터 누수)** 가 발생하고, 실제 서비스에서는 test 데이터를 미리 볼 수 없다는 전제 자체가 깨집니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# ❌ 잘못된 방법 (Data Leakage 위험)
wrong_test_scaled = StandardScaler().fit_transform(test)   # test로 새로 fit! (기준이 train과 달라짐)

print("올바른 방법(test.transform):", test_scaled[:3].round(2))
print("잘못된 방법(test.fit_transform):", wrong_test_scaled[:3].round(2))
# 두 결과가 다름! — 실무에서는 항상 train 기준을 재사용해야 함
```

</details>

In [ ]:
# ❌ 잘못된 방법 (Data Leakage 위험)
wrong_test_scaled = StandardScaler().fit_transform(test)   # test로 새로 fit! (기준이 train과 달라짐)

print("올바른 방법(test.transform):", test_scaled[:3].round(2))
print("잘못된 방법(test.fit_transform):", wrong_test_scaled[:3].round(2))
# 두 결과가 다름! — 실무에서는 항상 train 기준을 재사용해야 함

### 6) 서로 다른 방식 비교 — MinMax vs Standard vs Robust

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

qty = df[["수량"]]   # 이상치(40,50,60)가 포함된 컬럼으로 비교

minmax_result = MinMaxScaler().fit_transform(qty)
print("MinMax 결과 범위:", minmax_result.min(), "~", minmax_result.max())
# 이상치 때문에 정상 값(1~5)들이 전부 0에 가깝게 뭉쳐버림!
```

</details>

In [ ]:
# 코드 입력

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

qty = df[["수량"]]   # 이상치(40,50,60)가 포함된 컬럼으로 비교

minmax_result = MinMaxScaler().fit_transform(qty)
print("MinMax 결과 범위:", minmax_result.min(), "~", minmax_result.max())

minmax_result[:10]
# 이상치 때문에 정상 값(1~5)들이 전부 0에 가깝게 뭉쳐버림!

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
standard_result = StandardScaler().fit_transform(qty)
robust_result = RobustScaler().fit_transform(qty)

print("Standard 결과 (이상치 60의 스케일된 값):", standard_result[df["수량"]==60][:1])
print("Robust 결과 (이상치 60의 스케일된 값):", robust_result[df["수량"]==60][:1])
# Robust는 이상치의 영향을 덜 받아 정상 값들의 구분력이 더 잘 유지됨
```

</details>

In [ ]:
# 코드 입력

standard_result = StandardScaler().fit_transform(qty)
robust_result = RobustScaler().fit_transform(qty)

print("Standard 결과 (이상치 60의 스케일된 값):", standard_result[df["수량"]==60][:1])
print("Robust 결과 (이상치 60의 스케일된 값):", robust_result[df["수량"]==60][:1])

# Robust는 이상치의 영향을 덜 받아 정상 값들의 구분력이 더 잘 유지됨

result = df[["수량"]].copy()
result["standard"] = standard_result
result["robust"] = robust_result

print(result[result["수량"].isin([1, 2, 3, 4, 5, 40, 50, 60])]
      .groupby("수량")
      .first())

| 구분 | MinMaxScaler | StandardScaler | RobustScaler |
|---|---|---|---|
| 기준 | 최솟값·최댓값 | 평균·표준편차 | 중앙값·IQR |
| 결과 범위 | 정확히 0~1 | 평균 0, 표준편차 1 | 범위 고정 없음 |
| 이상치 영향 | 매우 민감 (정상값이 뭉침) | 민감 | 상대적으로 안정적 |
| 언제 사용? | 이상치가 없고 범위를 0~1로 고정하고 싶을 때 | 정규분포에 가까운 데이터, 딥러닝/회귀 | 이상치가 있는 데이터 |

> 💡 **핵심 요약**: 이상치가 없다고 확신하면 MinMax나 Standard를, CH09에서처럼 이상치가 확인된 데이터(수량 등)라면 RobustScaler를 우선 고려하세요.

---

## ⚠️ 자주 하는 실수

> ⚠️ 훈련 데이터와 테스트 데이터를 나눈 뒤, 스케일링을 나누기 **전에** 전체 데이터에 `fit_transform()`을 적용하는 경우가 있습니다 — 이것도 Data Leakage입니다. 반드시 분리 후 train만으로 `fit()`해야 합니다.

> ⚠️ 이상치가 있는 데이터에 MinMaxScaler를 그대로 적용해서, 정상 값들이 전부 0 근처로 뭉쳐버리는 것을 알아채지 못하는 경우가 흔합니다.

> ⚠️ 인코딩된 범주형 컬럼(0/1)까지 스케일링 대상에 포함시키는 실수가 있습니다 — 스케일링은 수치형(연속값) 컬럼에만 적용합니다.

---

## 🚀 실무에서는?

> 🚀 실무에서는 데이터를 훈련/검증/테스트로 나눈 뒤, **훈련 데이터의 통계량으로 학습한 스케일러**를 `pickle` 등으로 저장해두고 실제 서비스에서도 동일한 스케일러를 재사용합니다. 이상치가 의심되면 CH09에서 배운 IQR/Z-score로 먼저 점검한 뒤 스케일러 종류를 선택하는 것이 정석입니다.

---

## 🧩 Check Point

**Q1.** 값의 범위를 정확히 0~1로 고정하는 스케일러는? ① StandardScaler ② MinMaxScaler ③ RobustScaler ④ LabelEncoder
<details><summary>정답</summary>② MinMaxScaler</details>

**Q2.** 이상치의 영향을 가장 덜 받는 스케일러는? ① MinMaxScaler ② StandardScaler ③ RobustScaler ④ 모두 동일
<details><summary>정답</summary>③ RobustScaler (중앙값·IQR 기반)</details>

**Q3.** 테스트 데이터에 스케일링을 적용할 때 올바른 방법은? ① 테스트 데이터로 새로 fit_transform() ② 훈련 데이터의 스케일러로 transform()만 ③ 스케일링 불필요 ④ 둘 다 fit_transform()
<details><summary>정답</summary>② 훈련 데이터의 스케일러로 transform()만 (Data Leakage 방지)</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (MinMaxScaler): "단가" 컬럼을 MinMaxScaler로 0~1 사이로 변환하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df["단가_mm"] = scaler.fit_transform(df[["단가"]])
print(df[["단가", "단가_mm"]].head())
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 2 (StandardScaler): "배송평점" 컬럼을 표준화하고 평균/표준편차를 확인하세요 (결측치 dropna 먼저)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
from sklearn.preprocessing import StandardScaler
rating = df["배송평점"].dropna().to_frame()
scaler2 = StandardScaler()
rating["평점_std"] = scaler2.fit_transform(rating[["배송평점"]])
print(rating["평점_std"].mean().round(3), rating["평점_std"].std().round(3))
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 3 (RobustScaler+이상치): "수량"에 RobustScaler와 MinMaxScaler를 각각 적용해
#         이상치(40,50,60)가 있는 값의 스케일된 결과를 비교하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
from sklearn.preprocessing import RobustScaler
mm = MinMaxScaler().fit_transform(df[["수량"]])
rb = RobustScaler().fit_transform(df[["수량"]])
print("MinMax 결과 (이상치 포함) 범위:", mm.min(), "~", mm.max())
print("Robust 결과 (이상치 포함) 범위:", rb.min(), "~", rb.max())
```

</details>

In [ ]:
# 코드 입력





In [ ]:
# TODO 4 (train/test): 데이터를 앞 250개(train)/뒤(test)로 나눠
#         올바른 방식(train만 fit)으로 "단가"를 스케일링하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
train = df[["단가"]].iloc[:250]
test = df[["단가"]].iloc[250:]
scaler5 = StandardScaler()
train_s = scaler5.fit_transform(train)
test_s = scaler5.transform(test)
print(train_s[:3])
print(test_s[:3])
```

</details>

In [ ]:
# 코드 입력





---

## 📝 실습 과제

In [ ]:
# 과제 1: 배송평점 [1, 2, 3, 4, 5, 55]에 이상치(55)가 포함되어 있을 때,
#         MinMaxScaler와 RobustScaler 결과를 비교해 어떤 스케일러가 더 안정적인지 설명하세요

In [ ]:
# 과제 2: "단가"와 "수량" 두 컬럼을 동시에 StandardScaler로 스케일링하고,
#         스케일링 전후의 각 컬럼 표준편차를 비교하세요

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
ratings = pd.DataFrame({"평점": [1, 2, 3, 4, 5, 55]})
print("MinMax:", MinMaxScaler().fit_transform(ratings).flatten().round(3))
print("Robust:", RobustScaler().fit_transform(ratings).flatten().round(3))
# MinMax는 55 때문에 1~5가 전부 0 근처로 뭉치지만, Robust는 상대적으로 값의 구분이 유지됨
```

</details>

In [ ]:
# 과제 1 예시 정답




# MinMax는 55 때문에 1~5가 전부 0 근처로 뭉치지만, Robust는 상대적으로 값의 구분이 유지됨

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
scaler6 = StandardScaler()
scaled = scaler6.fit_transform(df[["단가", "수량"]])
print("스케일링 전 표준편차:", df[["단가", "수량"]].std().values.round(2))
print("스케일링 후 표준편차:", scaled.std(axis=0).round(2))   # 둘 다 1에 가까워짐
```

</details>

In [ ]:
# 과제 2 예시 정답




---

## 📌 핵심 정리

- 스케일링은 값의 범위가 다른 컬럼들을 비슷한 규모로 맞추는 전처리 과정
- `MinMaxScaler`: 0~1로 고정, 이상치에 민감
- `StandardScaler`: 평균 0·표준편차 1, 정규분포에 적합
- `RobustScaler`: 중앙값·IQR 기반, 이상치에 강함
- train 데이터로 `fit()`(또는 `fit_transform()`)하고, test 데이터는 반드시 `transform()`만 — Data Leakage 방지
- 스케일링은 수치형 컬럼에만 적용 (인코딩된 0/1 컬럼은 대상 아님)

---

---

# 📘 CH10-03. 웹크롤링 프로젝트 — 미스터리 소설 전체 목록 수집

━━━━━━━━━━━━━━━━

> 🗂️ **프로젝트 목표**: `books.toscrape.com`의 Mystery 카테고리에 있는 책 32권 전체를 자동으로 수집해 DataFrame과 CSV로 정리한다
> 🔧 **실습파일**: 불필요 (인터넷에서 직접 수집)
> ⏱️ **예상 소요시간**: 70분
> 🎚️ **난이도**: ★★★★★
> 🔗 **사전 지식**: CH02-04-04(JSON), 인터넷 연결 필요
> ⚠️ 이번 소단원은 TODO/실습 과제 없이, 하나의 프로젝트를 처음부터 끝까지 함께 만들어봅니다.

---

## 🤔 먼저 생각해보기

> **온라인 서점에서 특정 장르의 책 32권을 제목·가격·평점까지 일일이 손으로 복사-붙여넣기 한다면 얼마나 걸릴까요?**

이런 반복 작업을 자동화하는 것이 **웹크롤링**입니다. 이번 시간에는 실제로 존재하는 웹사이트에서 데이터를 끝까지 수집해 분석 가능한 형태로 만드는 **미니 프로젝트**를 진행합니다.

> 📌 `books.toscrape.com`은 웹크롤링 연습을 위해 만들어진 공개 사이트로, 사이트 첫 화면에 "We love being scraped!(우리는 스크래핑되는 걸 좋아해요!)"라고 적혀 있을 만큼 크롤링 연습용으로 안전하게 사용할 수 있습니다.

---

## 📖 프로젝트 진행 순서

1. 사이트에 연결이 되는지 확인한다
2. HTML을 파싱해서 책 한 권의 정보를 뽑아본다
3. 페이지에 있는 책 전체 정보를 뽑는 함수로 만든다
4. 다음 페이지로 자동으로 넘어가며 전체 페이지를 순회한다
5. 결과를 DataFrame으로 정리하고, 가격/평점을 분석 가능한 형태로 다듬는다
6. 간단히 분석하고 시각화한다
7. 최종 결과를 CSV로 저장한다

---

## Step 1. 사이트 연결 확인

In [ ]:
import requests

url = "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"

# 지정한 URL에 요청(get) 보내기




# 응답 상태 코드 확인 (200: 정상, 404: 없음, 500: 서버오류)






> 💡 `status_code`가 200이 아니면 페이지를 제대로 가져오지 못한 것이므로, 이후 코드를 실행하기 전에 항상 확인하는 습관이 중요합니다. (403: 접근 거부, 404: 페이지 없음, 500: 서버 오류)

## Step 2. HTML 파싱하기 — BeautifulSoup

> 파싱(Parsing)이란 필요한 정보를 읽어서 구조화하는 것
- 데이터를 분석해서 원하는 정보를 쉽게 사용할 수 있는 형태로 바꾸는 과정입니다.

In [ ]:
# 라이브러리 불러오기


# 웹 사이트에서 받아온 HTMl 원본 준비



# 페이지 제목 확인 — 잘 파싱됐는지 1차 점검
# title 태그의 문자만 가지고오기




# 첫 500글자만 보기





## Step 3. 책 한 권의 정보 뽑아보기

- div → 그냥 박스 (가장 많이 씀)
- section → 큰 구역
- article → 독립된 콘텐츠 하나 (게시글, 뉴스, 상품 등)
- ol → 목록
- li → 목록의 한 칸
- h3 → 제목
- a → 링크
- p → 글(문단)

- `<section>` : 책 목록 전체 영역
- `<div>` : 영역을 나누는 박스
- `<ol class = 'row'>` :책 목록 (리스트) 시작
- `<li class = 'col-xs-6 col-sm-4 col-md-3 col-lg-3'>`: 책 1권을 담는 칸 (이 리스트가 20개)

- `<article class="product_pod">` : 책 1권의 실제 정보 시작
- `<div class="image_container">` : 책 표지 이미지 영역
- `<p class="star-rating Four">` : 평점
- `<H3>` : 책 제목 영역
- `<a href="..." title="Sharp Objects">` : 책 상세페이지 링크 + 책 제목
- `<div class="product_price">` : 가격과 재고 정보 영역



```
section
 └── ol
      └── li
           └── article   ← 책 1권
                ├── div 이미지
                ├── h3  제목
                    └── a 링크 + 제목
                ├── div 가격
                    └── p 가격
                    └── p 재고
                └── p   평점
```



<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 첫 번째 책 하나만 먼저 파싱해보고, 구조를 파악한다

# 첫 번재 책정보 가져오기 (find_all은 조건에 맞는 모든 책 가져오기)
# soup.find(태그이름, 조건) # 첫번째 article을 찾아옴

first_book = soup.find("article", class_="product_pod")

# 제목은 <a> 태그의 title 속성에 있는걸 가지고 옴
title = first_book.h3.a['title']

# select_one()은 CSS 선택자를 이용해서 원하는 태그 하나를 찾는 함수
# 가격은 "£47.82" 형태
#price = first_book.find("p", class_="price_color").text (find 도 동일하게 작성 가능)

# class= 'price_color'인 p 태그에서 가격 가져오기
price = first_book.select_one("p.price_color").text

# class='instock availability'인 p 태그에서 재고 상태 가져오기 (클래스가 2개 (2가지 정보 노출))
availability = first_book.select_one("p.instock.availability").text.strip()

# p 클래스 이름이 'star-rating' 인것의 클래스를 리스트로 반환
rating_class = first_book.select_one("p.star-rating")["class"]

print("제목:", title)
print("가격:", price)
print("재고:", availability)
print("평점 class:", rating_class)   # 두 번째 값이 "One"~"Five" 텍스트로 된 평점
```

</details>

In [ ]:
# 첫 번째 책 하나만 먼저 파싱해보고, 구조를 파악한다

# 첫 번재 책정보 가져오기







> 💡 별점은 숫자가 아니라 `star-rating Three`처럼 **CSS 클래스명(One~Five)**으로 표현되어 있습니다. 이런 사이트 고유의 구조는 실제로 페이지를 열어 개발자 도구(F12)로 직접 확인하며 파악해야 합니다.

## Step 4. 함수로 만들기 — 한 페이지의 책 전체 정보 추출

In [ ]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_page(soup):
    """한 페이지의 BeautifulSoup 객체를 받아 책 정보 리스트를 반환"""
    books = soup.find_all("article", class_="product_pod")

    result = []

    for book in books:

        title = book.h3.a["title"]
        price = float(book.select_one("p.price_color").text.replace("£", ""))
        rating_word = book.select_one("p.star-rating")["class"][1]
        rating = RATING_MAP.get(rating_word, None)
        availability = book.select_one("p.instock.availability").text.strip()
        result.append({"제목": title, "가격": price, "평점": rating, "재고상태": availability})
    return result

# 방금 가져온 페이지로 테스트
page1_books = parse_page(soup)
print(len(page1_books), "권 추출됨")
print(page1_books[:3])

## Step 5. 다음 페이지로 자동 이동 — 페이지네이션

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# "Page 1 of 2" 같은 표시 아래에 다음 페이지로 가는 링크가 있는지 확인

# class='next'인 li 태그안에 있는 a를 찾아라.
next_link = soup.select_one("li.next a")

# 위에서 찾은 a 태그 출력
print(next_link)

# 다음 페이지의 상대 경로, 예: "page-2.html"
if next_link:
    print(next_link["href"])
```

</details>

In [ ]:
# "Page 1 of 2" 같은 표시 아래에 다음 페이지로 가는 링크가 있는지 확인

# class='next'인 li 태그안에 있는 a를 찾아라.




# 위에서 찾은 a 태그 출력




# 다음 페이지의 상대 경로, 예: "page-2.html"





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import time

# 카테고리 페이지의 기본 URL (여기에 page-N.html을 이어붙여서 페이지를 이동)
category_url = "https://books.toscrape.com/catalogue/category/books/mystery_3/"

all_books = []
# 첫번째 페이지 (index.html)부터 크롤링 시작
page_url = category_url + "index.html"

# 다음페이지가 없을 때까지 반복
while True:
    res = requests.get(page_url)
    res.encoding = 'utf-8'
    if res.status_code != 200:
        print("요청 실패:", res.status_code)
        break

    page_soup = BeautifulSoup(res.text, "html.parser")
    all_books.extend(parse_page(page_soup)) # list에 이어 붙이기
    print(f"{page_url} → 누적 {len(all_books)}권") # 현재페이지 + 누적개수 출력

    next_a = page_soup.select_one("li.next a") # 다음페이지 링크 찾기
    if next_a is None: # 다음 페이지 없으면 종료
        break                                   # 다음 페이지가 없으면 종료
    page_url = category_url + next_a["href"]     # 다음 페이지 URL로 갱신

    time.sleep(1)     # 서버 부담을 줄이기 위해 요청 사이에 1초씩 쉬어주는 것이 예의

print("\n총", len(all_books), "권 수집 완료")
```

</details>

In [ ]:
import time

# 카테고리 페이지의 기본 URL (여기에 page-N.html을 이어붙여서 페이지를 이동)
category_url = "https://books.toscrape.com/catalogue/category/books/mystery_3/"






> 💡 페이지가 몇 개인지 미리 알 필요 없이, "다음(next) 링크가 있는가?"를 기준으로 반복문을 종료하는 방식이 실전에서 가장 안전합니다. 카테고리마다 책 권수(페이지 수)가 달라도 코드를 수정할 필요가 없습니다.
> ⚠️ 요청 사이에 `time.sleep()`을 넣지 않고 반복문을 빠르게 돌리면 짧은 시간에 서버에 과도한 부담을 줄 수 있습니다 — 상대방 서버를 배려하는 최소한의 예의입니다.

## Step 6. 결과를 DataFrame으로 정리

In [ ]:
import pandas as pd

books_df = pd.DataFrame(all_books)
print(books_df.shape)  # (32, 4) — Mystery 카테고리 전체 책 32권
books_df.head()

## Step 7. 간단히 분석해보기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 평점별 책 권수
print(books_df["평점"].value_counts().sort_index())

# 가격 통계
print(books_df["가격"].describe())

print("\n가장 비싼 책 Top 3")
print(books_df.sort_values("가격", ascending=False).head(3)[["제목", "가격"]])

print("\n평점별 평균 가격")
print(books_df.groupby("평점")["가격"].mean().round(2))
```

</details>

In [ ]:
# 평점별 책 권수



# 가격 통계





<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 도서 가격 분포
axes[0].hist(books_df["가격"], bins=8, color="skyblue", edgecolor="black")
axes[0].set_title("Mystery 도서 가격 분포")
axes[0].set_xlabel("가격 (£)")

# 평점 별 책 권수
books_df["평점"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="tomato")
axes[1].set_title("평점별 책 권수")
axes[1].set_xlabel("평점")

plt.tight_layout()
plt.show()
```

</details>

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 도서 가격 분포



# 평점 별 책 권수




plt.tight_layout()
plt.show()

## Step 8. 최종 결과 저장

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
books_df.to_csv("mystery_books.csv", index=False, encoding="utf-8-sig")
print("mystery_books.csv 저장 완료")
print(books_df.head())
```

</details>

In [ ]:
# 코드 입력





---

## 📌 핵심 정리 — 웹크롤링 프로젝트 전체 파이프라인

1. `requests.get(url)`로 페이지를 가져오고 `status_code`로 성공 여부를 확인한다
2. `BeautifulSoup(response.text, "html.parser")`로 HTML을 파싱한다
3. `find()`/`find_all()`, `select_one()`/`select()`로 원하는 태그·클래스를 찾는다
4. 반복되는 추출 로직은 함수로 만들어 페이지마다 재사용한다
5. "다음 페이지 링크가 있는가?"를 기준으로 반복문을 돌며 전체 페이지를 순회하고, 요청 사이에 `time.sleep()`으로 서버를 배려한다
6. 수집한 결과를 `pd.DataFrame()`으로 정리하고, 문자열(가격·평점)을 분석 가능한 숫자로 변환한다
7. `groupby()`, 시각화 등 지금까지 배운 도구로 분석하고, `to_csv()`로 저장한다

> 다음 소단원(robots.txt와 크롤링 윤리)에서는 지금 크롤링한 `books.toscrape.com`이 실제로 이런 크롤링을 허용하는 사이트인지 공식적으로 확인하는 방법을 배웁니다.

---

---

# 📘 CH10-04. OpenAPI와 JSON 활용

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: API란? → 지오코딩 API로 좌표 조회 → 날씨 예보 API 호출 → JSON 파싱 → 여러 도시 비교 → 웹크롤링 vs API 비교
> 🔧 **실습파일**: 불필요 (실제 무료 공개 API 사용, 인증키 불필요)
> ⏱️ **예상 소요시간**: 45분
> 🎚️ **난이도**: ★★★☆☆
> 🔗 **사전 지식**: CH10-03(웹크롤링), 인터넷 연결 필요

---

## 🤔 먼저 생각해보기

> **날씨 앱은 매번 기상청 홈페이지를 크롤링해서 화면을 긁어올까요, 아니면 더 깔끔한 방법이 있을까요?**

많은 서비스(날씨, 지도, 환율 등)는 데이터를 필요한 사람이 쉽게 가져다 쓸 수 있도록 **API(정해진 형식으로 데이터를 주고받는 통로)** 를 공개합니다. 이번 시간에는 인증키 없이 누구나 쓸 수 있는 실제 날씨 API, **Open-Meteo**를 사용합니다.

---

## 📖 핵심 개념

### 1) API란? — 정해진 형식(JSON)으로 데이터를 주고받는 공식 통로

- API는 "질문을 보내면(JSON을 요청하면), 서버가 처리해서 결과(JSON)를 다시 보내주는 서비스"입니다.

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import requests

# Open-Meteo 지오코딩 API: 도시 이름을 위도·경도로 변환
url = "https://geocoding-api.open-meteo.com/v1/search"
params = {"name": "Seoul", "count": 1, "language": "en", "format": "json"}

response = requests.get(url, params=params)
print(response.status_code)   # 200이면 정상 응답
```

</details>

In [ ]:
# 코드 입력

import requests

# Open-Meteo 지오코딩 API: 도시 이름을 위도·경도로 변환
url = "https://geocoding-api.open-meteo.com/v1/search"

# API에 전달할 요청 정보 (파라미터)
params = {"name": "Seoul", "count": 1, "language": "en", "format": "json"}

# Seoul 정보 회신 바람 (URL끝에 붙여서 전달)
response = requests.get(url, params=params)

print(response.status_code)   # 200이면 정상 응답

> 💡 `params`에 딕셔너리를 전달하면 `requests`가 자동으로 `?name=Seoul&count=1...` 같은 쿼리스트링을 URL에 붙여줍니다. 크롤링과 달리 API는 처음부터 "이런 파라미터를 받는다"는 규격이 문서로 정해져 있습니다.

### 2) JSON 응답 파싱하기 — `response.json()`

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
data = response.json()   # JSON 문자열을 파이썬 딕셔너리로 자동 변환
print(type(data))
print(data["results"][0])          # 첫 번째 검색 결과 (서울의 좌표 정보)

seoul_lat = data["results"][0]["latitude"]
seoul_lon = data["results"][0]["longitude"]
print(f"서울 좌표: 위도 {seoul_lat}, 경도 {seoul_lon}")
```

</details>

In [ ]:
# 코드 입력

# JSON 문자열을 파이썬 딕셔너리로 자동 변환
# 정보를 json 형태로 받아옴
data = response.json()
print(type(data))

# 첫 번째 검색 결과 (서울의 좌표 정보)
print(data["results"][0])

seoul_lat = data["results"][0]["latitude"]
seoul_lon = data["results"][0]["longitude"]

print(f"서울 좌표: 위도 {seoul_lat}, 경도 {seoul_lon}")

### 3) 좌표로 날씨 예보 조회하기 — 두 API 연결하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
weather_url = "https://api.open-meteo.com/v1/forecast"
weather_params = {
    "latitude": seoul_lat,
    "longitude": seoul_lon,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "auto",
    "forecast_days": 7
}

weather_res = requests.get(weather_url, params=weather_params)
weather_data = weather_res.json()
print(weather_data["daily"])   # 날짜별 최고/최저기온, 강수량이 리스트로 들어있음
```

</details>

In [ ]:
# 날씨 정보를 제공하는 API 주소

weather_url = "https://api.open-meteo.com/v1/forecast"

# 서울의 위경도 알았으니, 직접 입력
# daily: 최고 기온, 최저기온, 하루 강수량 보내줘
# timezone: 해당 지역의 시간대로 자동 설정
# forecast_days: 7일
# API 가이드 참고하여 작성

weather_params = {
    "latitude": seoul_lat,
    "longitude": seoul_lon,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "auto",
    "forecast_days": 7
}

# 날씨 API에 요청 보냄!
weather_res = requests.get(weather_url, params=weather_params)

# 결과 받음
weather_data = weather_res.json()
print(weather_data["daily"])   # 날짜별 최고/최저기온, 강수량이 리스트로 들어있음

> 💡 지오코딩 API로 "도시 이름 → 좌표"를 구하고, 그 좌표를 다시 날씨 API에 넘기는 **2단계 API 연결**은 실무에서 매우 흔한 패턴입니다. 하나의 API가 다른 API의 입력값을 만들어주는 경우가 많습니다.

### 4) JSON을 DataFrame으로 변환하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import pandas as pd

forecast_df = pd.DataFrame({
    "날짜": weather_data["daily"]["time"],
    "최고기온": weather_data["daily"]["temperature_2m_max"],
    "최저기온": weather_data["daily"]["temperature_2m_min"],
    "강수량": weather_data["daily"]["precipitation_sum"],
})
print(forecast_df)
```

</details>

In [ ]:
# 코드 입력

import pandas as pd

forecast_df = pd.DataFrame({
    "날짜": weather_data["daily"]["time"],
    "최고기온": weather_data["daily"]["temperature_2m_max"],
    "최저기온": weather_data["daily"]["temperature_2m_min"],
    "강수량": weather_data["daily"]["precipitation_sum"],
})
print(forecast_df)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import matplotlib.pyplot as plt

plt.plot(forecast_df["날짜"], forecast_df["최고기온"], marker="o", label="최고기온")
plt.plot(forecast_df["날짜"], forecast_df["최저기온"], marker="o", label="최저기온")
plt.xticks(rotation=45)
plt.title("서울 7일 기온 예보")
plt.legend()
plt.tight_layout()
plt.show()
```

</details>

In [ ]:
# 코드 입력

import matplotlib.pyplot as plt

plt.plot(forecast_df["날짜"], forecast_df["최고기온"], marker="o", label="최고기온")
plt.plot(forecast_df["날짜"], forecast_df["최저기온"], marker="o", label="최저기온")
plt.xticks(rotation=45)
plt.title("서울 7일 기온 예보")
plt.legend()
plt.tight_layout()
plt.show()

### 5) 함수로 만들어 여러 도시 비교하기

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
def get_current_temp(city_name):
    """도시 이름을 받아 현재 기온을 반환하는 함수"""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                        params={"name": city_name, "count": 1}).json()
    if "results" not in geo:
        return None
    lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]

    weather = requests.get("https://api.open-meteo.com/v1/forecast",
                           params={"latitude": lat, "longitude": lon, "current": "temperature_2m"}).json()
    return weather["current"]["temperature_2m"]

cities = ["Seoul", "Tokyo", "Busan", "Jeju"]
temps = {city: get_current_temp(city) for city in cities}
print(temps)
```

</details>

In [ ]:
# 코드 입력

def get_current_temp(city_name):
    """도시 이름을 받아 현재 기온을 반환하는 함수"""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                        params={"name": city_name, "count": 1}).json()
    if "results" not in geo:
        return None
    lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]

    weather = requests.get("https://api.open-meteo.com/v1/forecast",
                           params={"latitude": lat, "longitude": lon, "current": "temperature_2m"}).json()
    return weather["current"]["temperature_2m"]

cities = ["Seoul", "Tokyo", "Busan", "Jeju"]
temps = {city: get_current_temp(city) for city in cities}
print(temps)

### 6) POST 요청으로 데이터 전송하기 (참고)

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 참고: 데이터를 "받아오는" GET과 달리, POST는 서버로 데이터를 "보낼" 때 사용
# 아래는 연습용 가짜 API(jsonplaceholder)로 POST 형식만 확인
new_post = {"title": "실습용 제목", "body": "실습용 내용", "userId": 1}
response_post = requests.post("https://jsonplaceholder.typicode.com/posts", json=new_post)
print(response_post.status_code)     # 201: 생성 성공
print(response_post.json())          # 서버가 만들어준 결과(가짜로 id가 붙어서 돌아옴)
```

</details>

In [ ]:
# 참고: 데이터를 "받아오는" GET과 달리, POST는 서버로 데이터를 "보낼" 때 사용
# 아래는 연습용 가짜 API(jsonplaceholder)로 POST 형식만 확인

new_post = {"title": "실습용 제목", "body": "실습용 내용", "userId": 1}
response_post = requests.post("https://jsonplaceholder.typicode.com/posts", json=new_post)
print(response_post.status_code)     # 201: 생성 성공
print(response_post.json())          # 서버가 만들어준 결과(가짜로 id가 붙어서 돌아옴)

### 7) 서로 다른 방식 비교 — 웹크롤링 vs API 활용

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 방법 A: 웹크롤링 — HTML 구조를 직접 분석해서 원하는 값을 찾아야 함 (CH10-03)
# from bs4 import BeautifulSoup
# soup = BeautifulSoup(requests.get(url).text, "html.parser")
# price = soup.select_one("p.price_color").text   # HTML 구조가 바뀌면 코드도 깨짐

# 방법 B: API 활용 — 정해진 형식(JSON)으로 바로 원하는 데이터를 받음
api_response = requests.get("https://api.open-meteo.com/v1/forecast",
                             params={"latitude": 37.57, "longitude": 126.98, "current": "temperature_2m"})
print(api_response.json()["current"]["temperature_2m"])   # 구조가 안정적이라 코드가 잘 안 깨짐
```

</details>

In [ ]:
# 코드 입력

# 방법 A: 웹크롤링 — HTML 구조를 직접 분석해서 원하는 값을 찾아야 함 (CH10-03)
# from bs4 import BeautifulSoup
# soup = BeautifulSoup(requests.get(url).text, "html.parser")
# price = soup.select_one("p.price_color").text   # HTML 구조가 바뀌면 코드도 깨짐

# 방법 B: API 활용 — 정해진 형식(JSON)으로 바로 원하는 데이터를 받음
api_response = requests.get("https://api.open-meteo.com/v1/forecast",
                             params={"latitude": 37.57, "longitude": 126.98, "current": "temperature_2m"})
print(api_response.json()["current"]["temperature_2m"])   # 구조가 안정적이라 코드가 잘 안 깨짐

| 구분 | 웹크롤링 | API 활용 |
|---|---|---|
| 데이터 형식 | HTML (구조를 직접 분석해야 함) | JSON (바로 사용 가능한 구조화된 데이터) |
| 안정성 | 웹사이트 디자인이 바뀌면 코드가 깨짐 | 공식 문서화된 규격이라 상대적으로 안정적 |
| 제공 여부 | 모든 웹사이트에서 가능(허용 시) | API를 공식 제공하는 곳에서만 가능 |
| 우선순위 | API가 없을 때의 대안 | 가능하다면 항상 API를 우선 고려 |

> 💡 **핵심 요약**: 원하는 데이터를 API로 공식 제공한다면 항상 API를 우선 사용하세요. 크롤링은 API가 없을 때의 대안입니다.

---

## ⚠️ 자주 하는 실수

> ⚠️ URL에 파라미터를 문자열로 직접 이어붙이는 경우가 있는데, `params` 딕셔너리를 사용하면 특수문자 인코딩 등을 `requests`가 자동으로 처리해줘서 훨씬 안전합니다.

> ⚠️ `response.json()`을 호출하기 전에 `status_code`를 확인하지 않아, 에러 응답(HTML 등)을 JSON으로 파싱하려다 에러가 나는 경우가 있습니다.

> ⚠️ 지오코딩 결과가 없을 수 있다는 것(도시 이름 오타 등)을 고려하지 않고 바로 `data["results"][0]`에 접근해 에러가 나는 경우가 있습니다 — 항상 존재 여부를 먼저 확인해야 합니다.

---

## 🚀 실무에서는?

> 🚀 실무에서는 대부분 공식 API를 최우선으로 찾아보고, API가 없거나 필요한 데이터를 제공하지 않을 때만 크롤링을 고려합니다. API는 호출 횟수 제한(rate limit)이 있는 경우가 많아 응답을 캐싱하거나 호출 빈도를 조절하는 것도 중요합니다.

---

## 🧩 Check Point

**Q1.** URL에 쿼리 파라미터를 안전하게 추가하는 requests의 옵션은? ① `data=` ② `params=` ③ `headers=` ④ `query=`
<details><summary>정답</summary>② `params=`</details>

**Q2.** API 응답(JSON)을 파이썬 딕셔너리로 바로 변환하는 메서드는? ① `response.text` ② `response.content` ③ `response.json()` ④ `response.dict()`
<details><summary>정답</summary>③ `response.json()`</details>

**Q3.** 웹크롤링과 비교했을 때 API 활용의 가장 큰 장점은? ① 항상 무료다 ② 데이터 형식이 안정적이고 바로 사용 가능하다 ③ 인증이 필요 없다 ④ 속도가 항상 빠르다
<details><summary>정답</summary>② 데이터 형식이 안정적이고 바로 사용 가능하다</details>

---

## 💻 실습

> 🔧 아래 코드 블록은 ipynb 셀로 그대로 변환됩니다. 난이도가 단계적으로 올라갑니다.

In [ ]:
# TODO 1 (지오코딩): "Busan"의 위도·경도를 조회하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                    params={"name": "Busan", "count": 1}).json()
print(geo["results"][0]["latitude"], geo["results"][0]["longitude"])
```

</details>

In [ ]:
geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                    params={"name": "Busan", "count": 1}).json()

# 코드 입력




In [ ]:
# TODO 2 (날씨 조회): 위 좌표로 현재 기온(current)을 조회하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]
w = requests.get("https://api.open-meteo.com/v1/forecast",
                 params={"latitude": lat, "longitude": lon, "current": "temperature_2m"}).json()
print(w["current"]["temperature_2m"])
```

</details>

In [ ]:
lat, lon = geo["results"][0]["latitude"], geo["results"][0]["longitude"]

# 코드 입력 (params 이후 부터)
w = requests.get("https://api.open-meteo.com/v1/forecast",
                 params=



In [ ]:
# TODO 3 (파라미터 확장): forecast_days=3, daily에 강수확률(precipitation_probability_max)을 추가해 조회하세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
w3 = requests.get("https://api.open-meteo.com/v1/forecast",
                  params={"latitude": lat, "longitude": lon,
                          "daily": "precipitation_probability_max",
                          "forecast_days": 3, "timezone": "auto"}).json()
print(w3["daily"])
```

</details>

In [ ]:
# 코드 입력 (params= 이후 부터)

w3 = requests.get("https://api.open-meteo.com/v1/forecast",
                  params=


In [ ]:
# TODO 4 (DataFrame 변환): 여러 도시(서울/부산/제주)의 현재 기온을 조회해 DataFrame으로 만드세요

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 예시 정답
rows = []
for city in ["Seoul", "Busan", "Jeju"]:
    t = get_current_temp(city)
    rows.append({"도시": city, "현재기온": t})
print(pd.DataFrame(rows))
```

</details>

In [ ]:
# 코드 입력





---

## 📝 실습 과제

In [ ]:
# 과제 1: "Incheon"의 7일 최고/최저기온을 조회해 DataFrame으로 만들고, 일교차(최고-최저)가 가장 큰 날을 찾으세요

In [ ]:
# 과제 2: 서울/도쿄/뉴욕 세 도시의 현재 기온을 조회해 막대그래프로 비교하세요
#         (힌트: get_current_temp 함수 재사용)

**예시 정답**

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 1 예시 정답
geo_i = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": "Incheon", "count": 1}).json()
lat_i, lon_i = geo_i["results"][0]["latitude"], geo_i["results"][0]["longitude"]
w_i = requests.get("https://api.open-meteo.com/v1/forecast",
                   params={"latitude": lat_i, "longitude": lon_i,
                           "daily": "temperature_2m_max,temperature_2m_min",
                           "timezone": "auto"}).json()
incheon_df = pd.DataFrame({
    "날짜": w_i["daily"]["time"],
    "최고기온": w_i["daily"]["temperature_2m_max"],
    "최저기온": w_i["daily"]["temperature_2m_min"],
})
incheon_df["일교차"] = incheon_df["최고기온"] - incheon_df["최저기온"]
print(incheon_df.sort_values("일교차", ascending=False).head(1))
```

</details>

In [ ]:
# 과제 1 예시 정답
# 실습 코드 참고




<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 과제 2 예시 정답
cities2 = ["Seoul", "Tokyo", "New York"]
temps2 = {city: get_current_temp(city) for city in cities2}
plt.bar(temps2.keys(), temps2.values(), color=["tomato", "skyblue", "gold"])
plt.title("도시별 현재 기온 비교")
plt.ylabel("기온 (°C)")
plt.show()
```

</details>

In [ ]:
# 과제 2 예시 정답






---

## 📌 핵심 정리

- API는 정해진 형식(주로 JSON)으로 데이터를 안정적으로 주고받는 공식 통로
- `requests.get(url, params={...})`로 쿼리 파라미터를 안전하게 담아 요청
- `response.json()`으로 JSON 응답을 파이썬 딕셔너리/리스트로 바로 변환
- 여러 API를 연결해서 사용하는 경우가 많음 (지오코딩 → 날씨 조회)
- `requests.post(url, json={...})`로 데이터 전송(생성) 요청 가능
- 원하는 데이터를 API로 제공한다면, 크롤링보다 API 사용을 우선 고려

---

---

# 📘 CH10-05. robots.txt와 크롤링 윤리

━━━━━━━━━━━━━━━━

> 🗂️ **챕터 구성**: robots.txt란? → robots.txt 구조 읽는 법 → 파이썬으로 robots.txt 확인하기 → 크롤링 윤리 원칙 → 허용된 크롤링 vs 금지된 크롤링 비교
> 🔧 **실습파일**: 불필요 (인터넷 연결 필요)
> ⏱️ **예상 소요시간**: 30분
> 🎚️ **난이도**: ★★★☆☆
> 🔗 **사전 지식**: CH10-03(웹크롤링)

---

## 🤔 먼저 생각해보기

> **아무 웹사이트나 원하는 대로 다 긁어와도 법적·윤리적으로 문제가 없을까요?**

크롤링이 기술적으로 "가능하다"는 것과 "해도 된다"는 것은 다른 문제입니다. 웹사이트 운영자는 `robots.txt`라는 파일로 "여기는 크롤링해도 되고, 여기는 안 된다"는 규칙을 명시합니다. 앞서 CH10-03에서 크롤링했던 `books.toscrape.com`이 실제로 이를 허용하는 사이트였는지, 이번 시간에 공식적으로 확인해봅니다.

---

## 📖 핵심 개념

### 1) `robots.txt`란?

- 웹사이트의 최상위 경로에 위치한 텍스트 파일 (예: `https://example.com/robots.txt`)
- 검색엔진이나 크롤러에게 "어떤 경로를 크롤링해도 되는지"를 알려주는 규칙 파일
- 법적 강제력은 약하지만, 웹의 오랜 관례이자 신뢰의 약속으로 반드시 지켜야 할 규칙으로 여겨짐

### 2) `robots.txt` 구조 읽는 법

```
User-agent: *
Disallow: /private/
Allow: /public/
Crawl-delay: 10

User-agent: Googlebot
Disallow: /admin/
```

| 지시어 | 의미 |
|---|---|
| `User-agent` | 이 규칙이 적용되는 대상 (`*`는 모든 크롤러, `Googlebot`은 구글봇만) |
| `Disallow` | 크롤링을 금지하는 경로 |
| `Allow` | (Disallow 중 일부를) 예외적으로 허용하는 경로 |
| `Crawl-delay` | 요청 사이에 지켜야 할 최소 대기 시간(초) |

> 위 예시는 "모든 크롤러는 `/private/` 경로는 크롤링 금지, `/public/`은 허용, 요청 사이 10초 대기"를, "구글봇은 추가로 `/admin/`도 금지"를 의미합니다.

### 3) 파이썬으로 `robots.txt` 확인하기

네이버 예시

| robots.txt 내용 | 의미 | 해석 |
|-----------------|------|------|
| `User-agent: *` | 적용 대상 | 모든 크롤러(검색엔진, 웹 크롤러 등)에 적용되는 규칙 |
| `Disallow: /` | 접근 금지 | 웹사이트의 모든 경로(`/`)를 크롤링하지 말라는 의미 |
| `Allow: /$` | 예외 허용 | 메인 페이지(`https://www.naver.com/`)만 크롤링 허용 (`$`는 문자열의 끝을 의미) |
| `Allow: /.we` | 특정 파일 허용 | 브라우저 및 서비스에서 사용하는 설정 파일은 크롤링 허용 |

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
import requests

response = requests.get("https://books.toscrape.com/robots.txt")
print(response.status_code)
print(response.text)   # robots.txt의 실제 내용 확인
```

</details>

In [ ]:
# 코드 입력

import requests

response = requests.get("https://naver.com/robots.txt")
print(response.status_code)
print(response.text)   # robots.txt의 실제 내용 확인

### 4) 크롤링 윤리 — robots.txt를 넘어선 고려사항

- **서버 부담 주지 않기**: `time.sleep()`으로 요청 간격을 두어 상대 서버에 과도한 부하를 주지 않음 (CH10-03에서 실제로 적용했습니다)
- **이용약관(Terms of Service) 확인**: robots.txt에 없어도 이용약관에서 크롤링을 명시적으로 금지할 수 있음
- **개인정보 수집 주의**: 이름, 연락처 등 개인을 식별할 수 있는 정보는 관련 법률(개인정보보호법 등)의 적용을 받을 수 있음
- **저작권 존중**: 수집한 콘텐츠(글, 이미지 등)를 무단으로 재배포하지 않음
- **과도한 트래픽 유발 금지**: 봇 여러 개로 동시에 무차별적으로 요청하는 행위는 서비스 거부 공격(DDoS)처럼 보일 수 있음

### 5) 서로 다른 상황 비교 — 허용된 크롤링 vs 문제가 되는 크롤링

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 상황 A: 허용된 크롤링 — robots.txt를 확인하고, 명시적으로 허용된 경로만, 적절한 속도로
from urllib.robotparser import RobotFileParser
import time

rp = RobotFileParser()
rp.set_url("https://books.toscrape.com/robots.txt")
rp.read()

url = "https://books.toscrape.com/catalogue/page-1.html"
if rp.can_fetch("*", url):
    response = requests.get(url)
    print("허용됨, 정상적으로 요청:", response.status_code)
    time.sleep(1)   # 다음 요청 전 대기
else:
    print("이 경로는 크롤링이 금지되어 있습니다")
```

</details>

In [ ]:
# 코드 입력

# 상황 A: 허용된 크롤링 — robots.txt를 확인하고, 명시적으로 허용된 경로만, 적절한 속도로
from urllib.robotparser import RobotFileParser
import time

rp = RobotFileParser()
rp.set_url("https://books.toscrape.com/robots.txt")
rp.read()

url = "https://books.toscrape.com/catalogue/page-1.html"
if rp.can_fetch("*", url):
    response = requests.get(url)
    print("허용됨, 정상적으로 요청:", response.status_code)
    time.sleep(1)   # 다음 요청 전 대기
else:
    print("이 경로는 크롤링이 금지되어 있습니다")

<details>
<summary> ---------- 💡 예시 코드 보기 ---------- </summary>

```python
# 상황 B: 문제가 될 수 있는 크롤링 — 확인 없이 금지된 경로를 빠르게 반복 요청
# for i in range(1000):
#     requests.get("https://example.com/private/data")   # robots.txt 확인도 없이, 대기도 없이 대량 요청
# → robots.txt 위반 + 서버 부담 + 잠재적 법적 문제 소지
```

</details>

In [ ]:
# 코드 입력

# 상황 B: 문제가 될 수 있는 크롤링 — 확인 없이 금지된 경로를 빠르게 반복 요청
for i in range(1000):
    requests.get("https://example.com/private/data")   # robots.txt 확인도 없이, 대기도 없이 대량 요청

# → robots.txt 위반 + 서버 부담 + 잠재적 법적 문제 소지

| 구분 | 허용된 크롤링 | 문제가 되는 크롤링 |
|---|---|---|
| robots.txt 확인 | 사전에 확인하고 준수 | 확인하지 않음 |
| 요청 속도 | `time.sleep()`으로 적절히 조절 | 짧은 시간에 대량 요청 |
| 수집 대상 | 공개된 일반 정보 | 개인정보, 저작권 콘텐츠 등 민감한 정보 |
| 활용 방식 | 개인 학습, 허용된 범위의 분석 | 무단 재배포, 상업적 도용 |

> 💡 **핵심 요약**: 크롤링을 시작하기 전에 항상 robots.txt를 확인하고, 요청 속도를 조절하고, 수집한 데이터의 활용 범위를 신중히 고려하세요. "기술적으로 가능하다"가 "해도 된다"를 의미하지 않습니다.

---

## ⚠️ 자주 하는 실수

> ⚠️ robots.txt를 아예 확인하지 않고 크롤링을 시작하는 경우가 매우 흔합니다.

> ⚠️ robots.txt만 준수하면 모든 크롤링이 법적으로 완전히 안전하다고 오해하는 경우가 있습니다 — 이용약관, 개인정보보호법 등 다른 고려사항도 함께 확인해야 합니다.

> ⚠️ `Crawl-delay`가 명시되어 있는데도 무시하고 빠르게 반복 요청하는 경우가 있습니다.

---

## 🚀 실무에서는?

> 🚀 실무에서 크롤링 기반 서비스를 운영할 때는 법무팀 검토를 거쳐 이용약관과 robots.txt를 함께 확인하는 것이 일반적입니다. 가능하다면 웹사이트가 공식 API를 제공하는지 먼저 확인하고(CH10-04), API가 없을 때만 신중하게 크롤링을 고려하는 것이 안전한 접근입니다.

---

## 🧩 Check Point

**Q1.** robots.txt에서 크롤링을 금지하는 경로를 나타내는 지시어는? ① Allow ② Disallow ③ User-agent ④ Crawl-delay
<details><summary>정답</summary>② Disallow</details>

**Q2.** 특정 URL의 크롤링 허용 여부를 True/False로 바로 확인해주는 파이썬 도구는? ① BeautifulSoup ② RobotFileParser ③ requests ④ json
<details><summary>정답</summary>② RobotFileParser (urllib.robotparser)</details>

---

---

## 📌 핵심 정리

- `robots.txt`는 크롤링 허용/금지 경로를 명시한 웹사이트의 규칙 파일
- `User-agent`, `Disallow`, `Allow`, `Crawl-delay` 지시어로 구성
- `urllib.robotparser.RobotFileParser`로 특정 URL의 크롤링 가능 여부를 코드로 확인 가능
- 크롤링 윤리: robots.txt 준수, 요청 속도 조절, 개인정보·저작권 주의, 이용약관 확인
- robots.txt 준수가 모든 법적 문제를 해결해주지는 않음 — 종합적인 판단 필요

---

---
## 🎉 정규 수업 완료!

인코딩, 스케일링, 웹크롤링 프로젝트, OpenAPI/JSON, robots.txt/크롤링윤리 5개 소단원을 모두 다뤘습니다. 인코딩·스케일링은 CH09에서 전처리한 `orders.csv`를 이어서 사용해 실전 감각을 살렸고, 웹크롤링은 실제 사이트(books.toscrape.com)를 대상으로 한 편의 프로젝트로, OpenAPI는 실제 무료 API(Open-Meteo)로 진행했습니다. 자율학습(심화: Selenium, 정규표현식, XML)은 관련 md 문서를 참고해주세요.
